# Getting started

In this notebook, we illustrate how to use the Neural News Recommendation with Multi-Head Self-Attention ([NRMS](https://aclanthology.org/D19-1671/)). The implementation is taken from the [recommenders](https://github.com/recommenders-team/recommenders) repository. We have simply stripped the model to keep it cleaner.

We use a small dataset, which is downloaded from [recsys.eb.dk](https://recsys.eb.dk/). All the datasets are stored in the folder path ```~/ebnerd_data/*```.

In [1]:
#Uncomment this section if you have the datasets in zip format.

#import zipfile
#import os
#
#def unzip_folder(zip_file_path, extract_to_dir):
#    if not os.path.exists(extract_to_dir):
#        os.makedirs(extract_to_dir)
#
#    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
#        zip_ref.extractall(extract_to_dir)
#
#unzip_folder('ebnerd_data/ebnerd_small.zip', 'ebnerd_data/ebnerd_small')
#unzip_folder('ebnerd_data/ebnerd_testset.zip', 'ebnerd_data/ebnerd_testset')


## Load functionality

In [24]:
!pip install "transformers>=4.30.0,<4.37.3"
!pip install "tensorflow>=2.12.0,<2.16.0"
!pip install "torch>=2.0.0,<2.3.0"
!pip install "scikit-learn==1.4.0"
!pip install "numpy>=1.24.0,<1.26.1"
!pip install "polars==0.20.8"
!pip install "pyyaml==6.0.1"
!pip install tqdm

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [25]:
from transformers import AutoTokenizer, AutoModel
from pathlib import Path
import tensorflow as tf
import polars as pl

from ebrec.utils._constants import (
    DEFAULT_HISTORY_ARTICLE_ID_COL,
    DEFAULT_CLICKED_ARTICLES_COL,
    DEFAULT_INVIEW_ARTICLES_COL,
    DEFAULT_IMPRESSION_ID_COL,
    DEFAULT_SUBTITLE_COL,
    DEFAULT_LABELS_COL,
    DEFAULT_TITLE_COL,
    DEFAULT_USER_COL,
)

from ebrec.utils._behaviors import (
    create_binary_labels_column,
    sampling_strategy_wu2019,
    add_known_user_column,
    add_prediction_scores,
    truncate_history,
)
from ebrec.evaluation import MetricEvaluator, AucScore, NdcgScore, MrrScore
from ebrec.utils._articles import convert_text2encoding_with_transformers
from ebrec.utils._polars import concat_str_columns, slice_join_dataframes
from ebrec.utils._articles import create_article_id_to_value_mapping
from ebrec.utils._nlp import get_transformers_word_embeddings
from ebrec.utils._python import write_submission_file, rank_predictions_by_score

from ebrec.models.newsrec.dataloader import NRMSDataLoader
from ebrec.models.newsrec.model_config import hparams_nrms
from ebrec.models.newsrec.nrms import NRMSModel

## Load dataset

In [26]:
def ebnerd_from_path(path: Path, history_size: int = 30) -> pl.DataFrame:
    """
    Load ebnerd - function
    """
    df_history = (
        pl.scan_parquet(path.joinpath("history.parquet"))
        .select(DEFAULT_USER_COL, DEFAULT_HISTORY_ARTICLE_ID_COL)
        .pipe(
            truncate_history,
            column=DEFAULT_HISTORY_ARTICLE_ID_COL,
            history_size=history_size,
            padding_value=0,
            enable_warning=False,
        )
    )
    df_behaviors = (
        pl.scan_parquet(path.joinpath("behaviors.parquet"))
        .collect()
        .pipe(
            slice_join_dataframes,
            df2=df_history.collect(),
            on=DEFAULT_USER_COL,
            how="left",
        )
    )
    return df_behaviors

### Generate labels
We sample a few just to get started. For testset we just make up a dummy column with 0 and 1 - this is not the true labels.

In [27]:
PATH = Path("ebnerd_data")
DATASPLIT = "ebnerd_small"

In this example we sample the dataset, just to keep it smaller. Also, one can simply add the testset similary to the validation.

In [30]:
COLUMNS = [
    DEFAULT_USER_COL,
    DEFAULT_HISTORY_ARTICLE_ID_COL,
    DEFAULT_CLICKED_ARTICLES_COL,
    DEFAULT_INVIEW_ARTICLES_COL,
    DEFAULT_IMPRESSION_ID_COL,
]
HISTORY_SIZE = 10
FRACTION = 0.01

df_train = (
    ebnerd_from_path(PATH.joinpath(DATASPLIT, "train"), history_size=HISTORY_SIZE)

    .pipe(
        sampling_strategy_wu2019,
        npratio=4,
        shuffle=True,
        with_replacement=True,
        seed=123,
    )
    .pipe(create_binary_labels_column)
)
# =>
df_validation = (
    ebnerd_from_path(PATH.joinpath(DATASPLIT, "validation"), history_size=HISTORY_SIZE)
    .select(COLUMNS)
    .pipe(create_binary_labels_column)
)

df_test = (
    ebnerd_from_path(Path(f"ebnerd_data/ebnerd_testset/ebnerd_testset/test"), history_size=HISTORY_SIZE)
    .with_columns(pl.Series(DEFAULT_CLICKED_ARTICLES_COL, [[]]))
    .pipe(
        sampling_strategy_wu2019,
        npratio=4,
        shuffle=True,
        with_replacement=True,
        seed=123,
    )
    .pipe(create_binary_labels_column)
    .sample(fraction=0.2)
)

df_test.head(10)


impression_id,impression_time,read_time,scroll_percentage,device_type,article_ids_inview,user_id,is_sso_user,gender,postcode,age,is_subscriber,session_id,is_beyond_accuracy,article_id_fixed,article_ids_clicked,labels
u32,datetime[μs],f32,f32,i8,list[i64],u32,bool,i8,i8,i8,bool,u32,bool,list[i32],list[i64],list[i8]
337406863,2023-06-02 05:21:36,69.0,100.0,2,"[9792843, null, … 9777983]",2189915,false,null,null,null,false,100858019,false,"[9770551, 9769472, … 9788184]",[null],"[0, 1, … 0]"
570849084,2023-06-04 08:00:57,66.0,100.0,1,"[9796271, 9744403, … 9796412]",1647953,false,null,null,null,false,14683031,false,"[9787524, 9790700, … 9790539]",[null],"[0, 0, … 0]"
572078148,2023-06-01 17:07:16,13.0,null,2,"[9792452, 9792452, … null]",500940,true,0,null,null,true,44589524,false,"[9789676, 9790085, … 9788393]",[null],"[0, 0, … 1]"
42968763,2023-06-01 19:33:54,48.0,null,3,"[9792131, null, … 9788041]",972654,false,null,null,null,false,77429234,false,"[9790548, 9789642, … 9789427]",[null],"[0, 1, … 0]"
48407836,2023-06-06 21:26:28,25.0,100.0,1,"[null, 9799896, … 9800084]",914310,false,null,null,null,false,33023530,false,"[9788107, 9788310, … 9791050]",[null],"[1, 0, … 0]"
100915347,2023-06-05 11:21:48,115.0,100.0,1,"[9798594, 9792128, … 9798534]",129678,true,null,null,null,false,96378961,false,"[9779248, 9782993, … 9788197]",[null],"[0, 0, … 0]"
157299718,2023-06-02 20:05:54,5.0,null,1,"[null, 7695717, … 7695717]",1407552,false,null,null,null,false,65397336,false,"[9790521, 9788151, … 9788352]",[null],"[1, 0, … 0]"
47584109,2023-06-04 08:55:38,66.0,null,2,"[9788643, 9796780, … 9793888]",2489835,false,null,null,null,false,83122261,false,"[9789664, 9790085, … 9790293]",[null],"[0, 0, … 0]"
455903331,2023-06-01 09:07:43,40.0,100.0,3,"[9791280, 9791154, … 9791374]",533043,false,null,null,null,false,88219725,false,"[9787524, 9787465, … 9789866]",[null],"[0, 0, … 0]"


## Load articles

In [ ]:
df_articles = pl.read_parquet(PATH.joinpath("ebnerd_small/articles.parquet"))
df_articles.head(2)

## Init model using HuggingFace's tokenizer and wordembedding
In the original implementation, they use the GloVe embeddings and tokenizer. To get going fast, we'll use a multilingual LLM from Hugging Face.
Utilizing the tokenizer to tokenize the articles and the word-embedding to init NRMS.


In [ ]:
TRANSFORMER_MODEL_NAME = "FacebookAI/xlm-roberta-base"
TEXT_COLUMNS_TO_USE = [DEFAULT_SUBTITLE_COL, DEFAULT_TITLE_COL]
MAX_TITLE_LENGTH = 30

# LOAD HUGGINGFACE:
transformer_model = AutoModel.from_pretrained(TRANSFORMER_MODEL_NAME)
transformer_tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL_NAME)

# We'll init the word embeddings using the
word2vec_embedding = get_transformers_word_embeddings(transformer_model)
#
df_articles, cat_cal = concat_str_columns(df_articles, columns=TEXT_COLUMNS_TO_USE)
df_articles, token_col_title = convert_text2encoding_with_transformers(
    df_articles, transformer_tokenizer, cat_cal, max_length=MAX_TITLE_LENGTH
)
# =>
article_mapping = create_article_id_to_value_mapping(
    df=df_articles, value_col=token_col_title
)

# Initiate the dataloaders
In the implementations we have disconnected the models and data. Hence, you should built a dataloader that fits your needs.

In [ ]:
train_dataloader = NRMSDataLoader(
    behaviors=df_train,
    article_dict=article_mapping,
    unknown_representation="zeros",
    history_column=DEFAULT_HISTORY_ARTICLE_ID_COL,
    eval_mode=False,
    batch_size=64
)
val_dataloader = NRMSDataLoader(
    behaviors=df_validation,
    article_dict=article_mapping,
    unknown_representation="zeros",
    history_column=DEFAULT_HISTORY_ARTICLE_ID_COL,
    eval_mode=True,
   batch_size=32
)
test_dataloader = NRMSDataLoader(
    behaviors=df_test,
    article_dict=article_mapping,
    unknown_representation="zeros",
    history_column=DEFAULT_HISTORY_ARTICLE_ID_COL,
    eval_mode=True,
    batch_size=32,
)

## Train the model


In [ ]:
MODEL_NAME = "NRMS"
LOG_DIR = f"downloads/runs/{MODEL_NAME}"
MODEL_WEIGHTS = f"downloads/data/state_dict/{MODEL_NAME}/weights"

# CALLBACKS
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=LOG_DIR, histogram_freq=1)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2)
modelcheckpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=MODEL_WEIGHTS, save_best_only=True, save_weights_only=True, verbose=1
)

hparams_nrms.history_size = HISTORY_SIZE
model = NRMSModel(
    hparams=hparams_nrms,
    word2vec_embedding=word2vec_embedding,
    seed=42,
)
hist = model.model.fit(
    train_dataloader,
    validation_data=val_dataloader,
    epochs=1,
    callbacks=[tensorboard_callback, early_stopping, modelcheckpoint],
)
_ = model.model.load_weights(filepath=MODEL_WEIGHTS)

# Example how to compute some metrics:

In [ ]:
pred_validation = model.scorer.predict(val_dataloader)

In [ ]:
pred_test = model.scorer.predict(test_dataloader)

## Add the predictions to the dataframe

In [ ]:
df_validation = add_prediction_scores(df_validation, pred_validation.tolist()).pipe(
    add_known_user_column, known_users=df_train[DEFAULT_USER_COL]
)
df_validation.head(5)

In [ ]:
df_test = add_prediction_scores(df_test, pred_test.tolist()).pipe(
    add_known_user_column, known_users=df_train[DEFAULT_USER_COL]
)
df_test.head(10)

### Compute metrics

In [ ]:
metrics = MetricEvaluator(
    labels=df_validation["labels"].to_list(),
    predictions=df_validation["scores"].to_list(),
    metric_functions=[AucScore(), MrrScore(), NdcgScore(k=5), NdcgScore(k=10)],
)
metrics.evaluate()

## Make submission file

In [ ]:
df_validation = df_validation.with_columns(
    pl.col("scores")
    .map_elements(lambda x: list(rank_predictions_by_score(x)))
    .alias("ranked_scores")
)
df_validation.head(5)

In [ ]:
df_test = df_test.with_columns(
    pl.col("scores")
    .map_elements(lambda x: list(rank_predictions_by_score(x)))
    .alias("ranked_scores")
)
df_test.head(5)

In [ ]:
import pandas as pd

# Assuming df_test is your Polars DataFrame
# Convert Polars DataFrame to Pandas DataFrame
df_pandas = df_test.to_pandas()

# Extract article_ids_inview and ranked_scores
df_selected = df_pandas[["article_ids_inview", "ranked_scores"]]

# Flatten the lists into separate columns
df_selected = pd.DataFrame({
    'article_ids_inview': df_selected['article_ids_inview'].tolist(),
    'ranked_scores': df_selected['ranked_scores'].tolist()
})

# Save to CSV
df_selected.to_csv('ebnerd_data/baseline_output.csv', index=False)

# Display the new dataframe
print(df_selected.head())


In [56]:
print(df_selected.head(10))

shape: (10, 2)
┌───────────────────────────────┬────────────────┐
│ article_ids_inview            ┆ ranked_scores  │
│ ---                           ┆ ---            │
│ list[i32]                     ┆ list[i64]      │
╞═══════════════════════════════╪════════════════╡
│ [9484153, 9525523, … 9785014] ┆ [6, 4, … 7]    │
│ [9793541, 9484153, … 9790040] ┆ [17, 15, … 9]  │
│ [9798805, 9798727, … 9798841] ┆ [1, 6, … 5]    │
│ [9788643, 9794321, … 9797114] ┆ [15, 17, … 12] │
│ [9797236, 9797371, … 9797287] ┆ [8, 3, … 7]    │
│ [9540802, 9797114, … 9797236] ┆ [2, 8, … 7]    │
│ [9799286, 9798109, … 9799411] ┆ [4, 3, … 6]    │
│ [9798093, 9798583, … 9745661] ┆ [3, 7, … 8]    │
│ [9474355, 9508223, … 9799797] ┆ [43, 45, … 7]  │
│ [9774648, 9774652, … 9770452] ┆ [3, 6, … 4]    │
└───────────────────────────────┴────────────────┘


This is using the validation, simply add the testset to your flow.

In [21]:
write_submission_file(
    impression_ids=df_validation[DEFAULT_IMPRESSION_ID_COL],
    prediction_scores=df_validation["ranked_scores"],
    path="downloads/predictions_baseline.txt",
)

25356it [01:06, 381.13it/s]


Zipping results/predictions_baseline.txt to results/predictions_baseline.zip


# DONE 🚀

In [31]:
df_test.write_parquet("ebnerd_data/ebnerd_testset/ebnerd_testset/reduced_testset.parquet")